# 淘宝用户活跃时间与留存分析

本 Notebook 基于淘宝 `UserBehavior.csv` 全量数据，使用 `chunksize` 分块读取方式完成用户活跃时间分析和新用户留存分析。

注意：原始数据约 3.67GB，本分析不会一次性读入全量数据，而是每次读取 1,000,000 行并累计统计结果。

## 一、导入库

先导入数据处理、可视化和累计统计需要用到的工具包。

In [ ]:
# 导入数据分析工具包
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import Counter, defaultdict

# 导入 Markdown 展示工具，用于在 Notebook 中展示动态业务解读
from IPython.display import Markdown, display

# 设置中文字体，避免图表中的中文显示为方框
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

# 设置 seaborn 的基础风格，让图表更清晰
sns.set_theme(style="whitegrid", font="SimHei")

## 二、设置路径

原始数据来自 E 盘项目目录，所有处理结果也保存到 E 盘项目的 `data/processed/` 目录。

In [ ]:
# 原始数据路径
raw_path = r"E:\ecommerce-user-growth-analysis\data\raw\UserBehavior.csv"

# 输出文件路径
hourly_output_path = r"E:\ecommerce-user-growth-analysis\data\processed\hourly_activity.csv"
weekday_output_path = r"E:\ecommerce-user-growth-analysis\data\processed\weekday_activity.csv"
daily_output_path = r"E:\ecommerce-user-growth-analysis\data\processed\daily_activity.csv"
retention_output_path = r"E:\ecommerce-user-growth-analysis\data\processed\user_retention.csv"

# 确保输出目录存在，如果不存在就自动创建
processed_dir = os.path.dirname(hourly_output_path)
os.makedirs(processed_dir, exist_ok=True)

# CSV 文件没有表头，因此手动指定列名
column_names = ["user_id", "item_id", "category_id", "behavior_type", "timestamp"]

# 分块大小：每次读取 100 万行，避免一次性占用过多内存
chunksize = 1000000

# 行为类型顺序，后续表格和图表都按这个顺序展示
behavior_order = ["pv", "fav", "cart", "buy"]

# 行为类型中文名称，方便图表展示
behavior_name_map = {
    "pv": "浏览",
    "fav": "收藏",
    "cart": "加购",
    "buy": "购买",
}

# 星期中文名称，dayofweek 中 0 表示周一，6 表示周日
weekday_name_map = {
    0: "周一",
    1: "周二",
    2: "周三",
    3: "周四",
    4: "周五",
    5: "周六",
    6: "周日",
}

print("原始数据路径：", raw_path)
print("处理后数据目录：", processed_dir)

## 三、分块读取全量数据

每个 chunk 读取后，先将时间戳转换为日期时间，再提取日期、小时和星期。无效时间会被删除，避免影响后续统计。

In [ ]:
# 如果原始文件不存在，提前报错，方便定位问题
if not os.path.exists(raw_path):
    raise FileNotFoundError(f"找不到原始数据文件：{raw_path}")

# 每小时总行为数，例如 {0: 100, 1: 80}
hour_total_counter = Counter()

# 每小时各行为数量，例如 {0: Counter({'pv': 90, 'buy': 10})}
hour_behavior_counter = defaultdict(Counter)

# 每周几总行为数，例如 {0: 1000, 1: 1200}
weekday_total_counter = Counter()

# 每周几各行为数量，例如 {0: Counter({'pv': 900, 'cart': 50})}
weekday_behavior_counter = defaultdict(Counter)

# 每日总行为数，例如 {date: 10000}
daily_total_counter = Counter()

# 每日各行为数量，例如 {date: Counter({'pv': 9000, 'buy': 100})}
daily_behavior_counter = defaultdict(Counter)

# 每日独立用户集合，用 set 对 user_id 去重
daily_user_sets = defaultdict(set)

# 每个用户第一次出现日期，用于计算新用户 cohort
user_first_date = {}

# 每个用户所有活跃日期，用于计算留存
user_active_dates = defaultdict(set)

# 记录处理进度
total_rows = 0
valid_rows = 0

# 使用 chunksize 分块读取全量数据，不一次性读入内存
reader = pd.read_csv(
    raw_path,
    names=column_names,
    header=None,
    chunksize=chunksize,
)

# 逐块处理数据
for chunk_no, chunk in enumerate(reader, start=1):
    # 累计原始读取行数
    total_rows += len(chunk)

    # 将 Unix 秒级时间戳转换为 datetime，无法转换的值会变成 NaT
    chunk["datetime"] = pd.to_datetime(chunk["timestamp"], unit="s", errors="coerce")

    # 删除 datetime 为空的数据，避免无效时间进入统计
    chunk = chunk.dropna(subset=["datetime"]).copy()

    # 新增日期字段，用于每日活跃和留存分析
    chunk["date"] = chunk["datetime"].dt.date

    # 新增小时字段，用于分析一天中哪个小时最活跃
    chunk["hour"] = chunk["datetime"].dt.hour

    # 新增星期字段，用于分析一周中哪几天更活跃
    chunk["weekday"] = chunk["datetime"].dt.dayofweek

    # 累计有效行数
    valid_rows += len(chunk)

    # 统计每小时总行为数
    hour_total_counter.update(chunk.groupby("hour").size().to_dict())

    # 统计每小时 pv、fav、cart、buy 各行为数量
    hour_behavior_counts = chunk.groupby(["hour", "behavior_type"]).size()
    for (hour, behavior_type), count in hour_behavior_counts.items():
        hour_behavior_counter[int(hour)][behavior_type] += int(count)

    # 统计每周几总行为数
    weekday_total_counter.update(chunk.groupby("weekday").size().to_dict())

    # 统计每周几 pv、fav、cart、buy 各行为数量
    weekday_behavior_counts = chunk.groupby(["weekday", "behavior_type"]).size()
    for (weekday, behavior_type), count in weekday_behavior_counts.items():
        weekday_behavior_counter[int(weekday)][behavior_type] += int(count)

    # 统计每日总行为数
    daily_total_counter.update(chunk.groupby("date").size().to_dict())

    # 统计每日 pv、fav、cart、buy 各行为数量
    daily_behavior_counts = chunk.groupby(["date", "behavior_type"]).size()
    for (date, behavior_type), count in daily_behavior_counts.items():
        daily_behavior_counter[date][behavior_type] += int(count)

    # 统计每日独立用户数：先按 date 和 user_id 去重，再更新每日用户集合
    date_user_pairs = chunk[["date", "user_id"]].drop_duplicates()
    for date, user_ids in date_user_pairs.groupby("date")["user_id"]:
        daily_user_sets[date].update(user_ids.tolist())

    # 留存分析需要知道每个用户在哪些日期活跃过
    user_date_pairs = chunk[["user_id", "date"]].drop_duplicates()
    for user_id, dates in user_date_pairs.groupby("user_id")["date"]:
        # 当前用户在这个分块中的活跃日期集合
        dates_set = set(dates.tolist())

        # 更新用户所有活跃日期
        user_active_dates[user_id].update(dates_set)

        # 更新用户首次出现日期
        min_date = min(dates_set)
        if user_id not in user_first_date or min_date < user_first_date[user_id]:
            user_first_date[user_id] = min_date

    # 打印处理进度，便于观察长任务是否正常运行
    print(f"已处理第 {chunk_no} 个分块，累计读取 {total_rows:,} 行，有效时间行 {valid_rows:,} 行")

print("\n全量数据分块读取完成")
print(f"累计读取行数：{total_rows:,}")
print(f"有效时间行数：{valid_rows:,}")
print(f"留存分析涉及用户数：{len(user_first_date):,}")

## 四、用户活跃时间分析

根据分块累计的统计结果，生成小时、星期和每日三个分析表。

In [ ]:
# 整理每小时活跃表
hourly_rows = []
for hour in range(24):
    # 每一行代表一个小时
    row = {
        "hour": hour,
        "total_behavior_count": int(hour_total_counter[hour]),
    }

    # 补充该小时四类行为数量
    for behavior_type in behavior_order:
        row[behavior_type] = int(hour_behavior_counter[hour][behavior_type])

    hourly_rows.append(row)

hourly_activity = pd.DataFrame(hourly_rows)

# 整理每周几活跃表
weekday_rows = []
for weekday in range(7):
    # 每一行代表一周中的一天
    row = {
        "weekday": weekday,
        "weekday_name": weekday_name_map[weekday],
        "total_behavior_count": int(weekday_total_counter[weekday]),
    }

    # 补充该星期四类行为数量
    for behavior_type in behavior_order:
        row[behavior_type] = int(weekday_behavior_counter[weekday][behavior_type])

    weekday_rows.append(row)

weekday_activity = pd.DataFrame(weekday_rows)

# 整理每日活跃表
daily_rows = []
all_dates = sorted(daily_total_counter.keys())
for date in all_dates:
    # 每一行代表一个自然日
    row = {
        "date": date,
        "total_behavior_count": int(daily_total_counter[date]),
        "uv": int(len(daily_user_sets[date])),
        "buy": int(daily_behavior_counter[date]["buy"]),
    }

    # 补充每日四类行为数量，其中 pv 常用于表示每日浏览量
    for behavior_type in behavior_order:
        row[behavior_type] = int(daily_behavior_counter[date][behavior_type])

    daily_rows.append(row)

daily_activity = pd.DataFrame(daily_rows)

# 将日期列转换成 pandas 日期类型，便于后续绘图
daily_activity["date"] = pd.to_datetime(daily_activity["date"])

# 展示三个结果表的前几行
display(hourly_activity.head())
display(weekday_activity)
display(daily_activity.head())

## 五、留存分析

留存分析以用户首次出现日期作为 cohort 日期。对于每个用户，计算其后续活跃日期距离首次出现日期的天数差，再统计次日、3 日和 7 日是否仍然活跃。

In [ ]:
# 新用户数：按首次出现日期统计用户数量
cohort_new_users = Counter(user_first_date.values())

# 留存用户数：例如 retained_users[cohort_date][1] 表示该 cohort 的次日留存用户数
retained_users = defaultdict(Counter)

# 遍历每个用户，判断其第 1、3、7 天是否仍活跃
for user_id, first_date in user_first_date.items():
    # 计算该用户所有活跃日期与首次出现日期的天数差
    active_day_offsets = {
        (active_date - first_date).days
        for active_date in user_active_dates[user_id]
    }

    # 如果第 1、3、7 天仍活跃，就计入对应留存人数
    for day in [1, 3, 7]:
        if day in active_day_offsets:
            retained_users[first_date][day] += 1

# 整理留存结果表
retention_rows = []
for cohort_date in sorted(cohort_new_users.keys()):
    # 当前 cohort 的首日新增用户数
    new_users = int(cohort_new_users[cohort_date])

    # 各留存周期的人数
    day_1_retained_users = int(retained_users[cohort_date][1])
    day_3_retained_users = int(retained_users[cohort_date][3])
    day_7_retained_users = int(retained_users[cohort_date][7])

    # 各留存周期的留存率
    retention_rows.append(
        {
            "cohort_date": cohort_date,
            "new_users": new_users,
            "day_1_retained_users": day_1_retained_users,
            "day_1_retention_rate": day_1_retained_users / new_users if new_users else 0,
            "day_3_retained_users": day_3_retained_users,
            "day_3_retention_rate": day_3_retained_users / new_users if new_users else 0,
            "day_7_retained_users": day_7_retained_users,
            "day_7_retention_rate": day_7_retained_users / new_users if new_users else 0,
        }
    )

user_retention = pd.DataFrame(retention_rows)

# 将 cohort_date 转换为日期类型，方便绘制时间序列图
user_retention["cohort_date"] = pd.to_datetime(user_retention["cohort_date"])

# 展示留存结果表
display(user_retention.head())

## 六、保存结果

将小时活跃、星期活跃、每日活跃和用户留存四张结果表保存到 E 盘项目的 `data/processed/` 目录。

In [ ]:
# 保存每小时活跃结果
hourly_activity.to_csv(hourly_output_path, index=False, encoding="utf-8-sig")

# 保存每周几活跃结果
weekday_activity.to_csv(weekday_output_path, index=False, encoding="utf-8-sig")

# 保存每日活跃结果
daily_activity.to_csv(daily_output_path, index=False, encoding="utf-8-sig")

# 保存用户留存结果
user_retention.to_csv(retention_output_path, index=False, encoding="utf-8-sig")

print("结果文件已保存：")
print(hourly_output_path)
print(weekday_output_path)
print(daily_output_path)
print(retention_output_path)

## 七、可视化

下面绘制 5 类图表，帮助观察用户在一天、一周和不同日期上的活跃规律，以及不同 cohort 的留存变化。

In [ ]:
# 1. 每小时总行为数折线图
plt.figure(figsize=(11, 5))
sns.lineplot(data=hourly_activity, x="hour", y="total_behavior_count", marker="o", color="#2F6B9A")
plt.title("每小时总行为数")
plt.xlabel("小时")
plt.ylabel("总行为数")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
# 2. 每小时不同类型行为数量折线图
plt.figure(figsize=(11, 5))
for behavior_type in behavior_order:
    sns.lineplot(
        data=hourly_activity,
        x="hour",
        y=behavior_type,
        marker="o",
        label=behavior_name_map[behavior_type],
    )

plt.title("每小时不同类型行为数量")
plt.xlabel("小时")
plt.ylabel("行为数量")
plt.xticks(range(24))
plt.legend(title="行为类型")
plt.tight_layout()
plt.show()

In [ ]:
# 3. 每周几总行为数柱状图
plt.figure(figsize=(9, 5))
sns.barplot(data=weekday_activity, x="weekday_name", y="total_behavior_count", color="#5F8D4E")
plt.title("每周几总行为数")
plt.xlabel("星期")
plt.ylabel("总行为数")
plt.tight_layout()
plt.show()

In [ ]:
# 4. 每日 PV、UV、购买数趋势图
# 三个指标量级差异较大，因此使用三个子图展示，避免小指标被大指标压扁
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

sns.lineplot(data=daily_activity, x="date", y="pv", marker="o", ax=axes[0], color="#2F6B9A")
axes[0].set_title("每日 PV 趋势")
axes[0].set_ylabel("PV")

sns.lineplot(data=daily_activity, x="date", y="uv", marker="o", ax=axes[1], color="#5F8D4E")
axes[1].set_title("每日 UV 趋势")
axes[1].set_ylabel("UV")

sns.lineplot(data=daily_activity, x="date", y="buy", marker="o", ax=axes[2], color="#B85C38")
axes[2].set_title("每日购买数趋势")
axes[2].set_xlabel("日期")
axes[2].set_ylabel("购买数")

plt.tight_layout()
plt.show()

In [ ]:
# 5. 次日、3 日、7 日留存率折线图
plt.figure(figsize=(12, 5))
sns.lineplot(data=user_retention, x="cohort_date", y="day_1_retention_rate", marker="o", label="次日留存率")
sns.lineplot(data=user_retention, x="cohort_date", y="day_3_retention_rate", marker="o", label="3 日留存率")
sns.lineplot(data=user_retention, x="cohort_date", y="day_7_retention_rate", marker="o", label="7 日留存率")

plt.title("新用户留存率趋势")
plt.xlabel("首次出现日期")
plt.ylabel("留存率")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.legend()
plt.tight_layout()
plt.show()

## 八、业务解读

本阶段只围绕用户活跃时间和用户留存进行分析，不做机器学习建模。下面的业务解读会基于前面计算出的实际结果自动生成。

In [ ]:
# 找出一天中总行为数最高的小时
peak_hour = int(hourly_activity.loc[hourly_activity["total_behavior_count"].idxmax(), "hour"])
peak_hour_count = int(hourly_activity["total_behavior_count"].max())

# 找出一周中总行为数最高的星期
top_weekdays = weekday_activity.sort_values("total_behavior_count", ascending=False).head(2)
top_weekday_text = "、".join(top_weekdays["weekday_name"].tolist())

# 找出浏览高峰和购买高峰对应的小时
pv_peak_hour = int(hourly_activity.loc[hourly_activity["pv"].idxmax(), "hour"])
buy_peak_hour = int(hourly_activity.loc[hourly_activity["buy"].idxmax(), "hour"])
peak_match_text = "一致" if pv_peak_hour == buy_peak_hour else "不完全一致"

# 计算平均留存率，便于概括整体留存表现
avg_day_1_retention = user_retention["day_1_retention_rate"].mean()
avg_day_3_retention = user_retention["day_3_retention_rate"].mean()
avg_day_7_retention = user_retention["day_7_retention_rate"].mean()

# 判断购买高峰是否集中在晚上，这里将 18 点到 23 点定义为晚上
evening_buy_text = "购买高峰集中在晚上" if 18 <= buy_peak_hour <= 23 else "购买高峰不集中在晚上"

# 使用 Markdown 展示业务解读
display(
    Markdown(
        f"""
### 1. 用户一天中哪个时间段最活跃？

从每小时总行为数看，用户最活跃的小时是 **{peak_hour}:00**，该小时总行为数为 **{peak_hour_count:,}**。运营上可以重点关注这个时段前后的流量承接、推荐位曝光和活动资源配置。

### 2. 用户一周中哪几天更活跃？

从每周几总行为数看，活跃度较高的是 **{top_weekday_text}**。如果这些日期对应周末或促销节点，说明用户更容易在空闲时间或活动刺激下产生浏览、加购和购买行为。

### 3. 浏览高峰和购买高峰是否一致？

浏览高峰出现在 **{pv_peak_hour}:00**，购买高峰出现在 **{buy_peak_hour}:00**，两者 **{peak_match_text}**。如果浏览高峰早于购买高峰，说明用户可能先浏览比较，再晚些时候下单；如果两者一致，说明高流量时段也能直接带动成交。

### 4. 留存率表现如何？

平均次日留存率约为 **{avg_day_1_retention:.2%}**，平均 3 日留存率约为 **{avg_day_3_retention:.2%}**，平均 7 日留存率约为 **{avg_day_7_retention:.2%}**。通常留存率会随着时间拉长而下降，如果下降过快，说明新用户首次访问后的持续回访动力不足。

### 5. 如果用户留存偏低，平台可以采取哪些运营策略？

如果留存偏低，可以从新用户首购激励、兴趣商品召回、收藏/加购提醒、个性化推荐、会员权益和内容化导购等方向优化。核心目标是让用户首次访问后有明确的再次访问理由。

### 6. 如果购买高峰集中在晚上，可以如何优化推荐、优惠券和活动推送？

当前判断结果是：**{evening_buy_text}**。如果购买高峰集中在晚上，可以在晚高峰前提前发放优惠券，在晚高峰期间强化首页推荐、购物车提醒和限时活动曝光，并将高意向用户的召回消息安排在购买高峰前 1 到 2 小时触达。

> 注意：数据集最后几天的 3 日和 7 日留存可能存在观察窗口不足的问题，解读时应重点参考观察窗口完整的 cohort。
        """
    )
)